# Agentic AI: Building AI Agents with Tools

This notebook introduces **agentic AI** - systems where LLMs can use tools, make decisions, and take actions autonomously.

## Learning Objectives
- Understand the agent architecture pattern
- Create custom tools for agents
- Build agents that can use multiple tools
- Handle tool errors and edge cases

## Related Theoretical Content
- [Multi-Agent AI](../../notes/04-multi_agent_ai/README.md)
- [Agent Architectures](../../notes/04-multi_agent_ai/01-agent_basics/README.md)
- [Tool Use and Function Calling](../../notes/03-ai/03-function_calling/README.md)

## Prerequisites
This notebook is self-contained and can run independently.

## Setup: Import Dependencies

Load required libraries and configure API access.

In [ ]:
import os
import random
import requests
import yfinance as yf
from io import BytesIO
from PIL import Image
from api_keys import get_api_key

# Configure environment
os.environ["TAVILY_API_KEY"] = get_api_key('tavily')

print(" Environment configured")

## 1. Create Custom Tools

**Tools** are functions that agents can call to interact with the world. LangChain's `@tool` decorator:
- Converts Python functions into agent-compatible tools
- Extracts descriptions from docstrings for the agent
- Handles type annotations for parameter validation

### Tool Design Best Practices:
1. **Clear docstrings**: Explain what the tool does and when to use it
2. **Type hints**: Help agents understand expected inputs
3. **Error handling**: Return informative messages instead of crashing
4. **Single responsibility**: Each tool should do one thing well

In [ ]:
from langchain.tools import tool
from langchain_tavily import TavilySearch

@tool
def web_search(query: str) -> str:
    """Use this tool to fetch latest info from the web for the query.
    Do not use this for fetching stock prices or financial data."""
    print(f'� Search tool invoked for: {query}')
    search = TavilySearch(max_results=3)
    return search.invoke(query)

@tool
def random_number_generator(start: int = 1, end: int = 100) -> int:
    """Use this tool to generate a random number between start and end."""
    print('� Random number generator invoked')
    return random.randint(start, end)

@tool
def get_current_user() -> str:
    """Get the current user's login name."""
    return os.getlogin()

@tool
def get_stock_price(ticker: str) -> str:
    """
    CRITICAL: Use this tool FIRST for any request involving stock prices, market caps, or volumes.
    If this does not yield results, do not use any other tool.
    Input must be a ticker symbol (e.g., MSFT, AAPL, GOOGL).
    """
    try:
        print(f"� Stock price fetcher invoked for: {ticker}")
        data = yf.Ticker(ticker)
        price = data.history(period="1d")['Close'].iloc[-1]
        return f"The current price for {ticker} is ${price:.2f}"
    except Exception as e:
        return f"Could not find price for {ticker}. Error: {str(e)}"

@tool
def generate_image(prompt: str) -> str:
    """Generates an image using the Pollinations AI engine.
    Returns a message indicating success or failure."""
    encoded_prompt = prompt.replace(" ", "%20")
    url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=1024&height=768&nologo=true"

    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            img = Image.open(BytesIO(response.content))

            # Save to data directory
            os.makedirs(".data", exist_ok=True)
            save_path = ".data/generated_image.png"
            img.save(save_path)

            return f"Image generated and saved to {save_path}"
        else:
            return "Failed to reach the image server."
    except Exception as e:
        return f"Error: {str(e)}"

print(" Custom tools created")

## 2. Initialize LLM

Choose an LLM that supports function calling. We'll use Groq's fast inference.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=get_api_key('groq')
)

print(" LLM initialized")

## 3. Create an Agent

**Agent Architecture:**
1. Receives user query
2. Decides which tool(s) to use
3. Calls tools with appropriate parameters
4. Synthesizes tool results into a response

The **system prompt** defines the agent's:
- Role and personality
- Behavior and constraints
- Tool usage guidelines

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Define agent system prompt
system_prompt = '''
You are an information expert and helpful assistant.

IMPORTANT INSTRUCTIONS:
- Always greet the current user by name (use get_current_user tool)
- Analyze user queries carefully and use the appropriate tools
- For stock prices, ALWAYS use get_stock_price tool first
- For general web information, use web_search
- Keep responses concise and under 40 lines
- If a tool fails, explain the failure clearly

Available tools:
- web_search: Fetch latest info from the web (NOT for stock prices)
- random_number_generator: Generate a random number
- generate_image: Generate an AI image from a text prompt
- get_current_user: Get the current user's name
- get_stock_price: Get current stock price for a ticker symbol
'''

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create agent with tools
tools = [
    web_search,
    random_number_generator,
    generate_image,
    get_current_user,
    get_stock_price
]

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print(" Agent created with 5 tools")

## 4. Test the Agent

Let's query the agent with different types of requests to see how it chooses and uses tools.

In [ ]:
def query_agent(question: str):
    """Query the agent with a question."""
    print(f"\n{'='*60}")
    print(f"User: {question}")
    print(f"{'='*60}\n")

    result = agent_executor.invoke({"input": question})

    print(f"\n{'='*60}")
    print(f"Agent: {result['output']}")
    print(f"{'='*60}\n")

    return result['output']

# Test 1: User information
query_agent("Who am I?")

In [ ]:
# Test 2: Stock price lookup
query_agent("What is the current price of Microsoft stock?")

In [ ]:
# Test 3: Web search
query_agent("What are the latest developments in AI in 2026?")

In [ ]:
# Test 4: Random number generation
query_agent("Give me a random number between 1 and 1000")

In [ ]:
# Test 5: Image generation
query_agent("Generate an image of a sunset over mountains")

In [ ]:
# Test 6: Complex query requiring multiple tools
query_agent("Hello! Can you tell me who I am and what's the stock price of Apple?")

## 5. Agent Decision Making

The agent demonstrates several intelligent behaviors:

1. **Tool Selection**: Chooses the right tool based on query intent
2. **Parameter Extraction**: Extracts ticker symbols, numbers, prompts from natural language
3. **Sequential Execution**: Calls multiple tools in logical order
4. **Error Recovery**: Handles tool failures gracefully
5. **Result Synthesis**: Combines tool outputs into coherent responses

## 6. Interactive Agent Loop

Create an interactive session where you can chat with the agent.

In [ ]:
def interactive_agent():
    """
    Run an interactive agent session.
    Type 'quit' or 'exit' to end the session.
    """
    print("� Agent Ready! (Type 'quit' to exit)\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break

        if user_input.strip():
            result = agent_executor.invoke({"input": user_input})
            print(f"\nAgent: {result['output']}\n")

# Uncomment to run interactive mode:
# interactive_agent()

## Key Takeaways

1. **Agent Architecture**: LLMs + Tools = Autonomous action capability
2. **Tool Design**: Clear interfaces and docstrings are critical
3. **System Prompts**: Define agent behavior and constraints
4. **Error Handling**: Tools should return informative errors, not crash
5. **Multi-Tool Reasoning**: Agents can chain multiple tools to solve complex tasks

## Limitations

- No memory between queries (stateless)
- Can't learn from past interactions
- Limited to predefined tools
- No long-term planning

## Next Steps

- [06-agent-memory.ipynb](06-agent-memory.ipynb): Add persistent memory to agents
- [07-langgraph.ipynb](07-langgraph.ipynb): Build complex agent workflows with LangGraph